# Phase 0 — Setup: Drive → Local Disk Pipeline (URI-CADS)

Goal: get the dataset from Drive onto local Colab disk **fast and reliably**, using the zip-on-Drive pattern (avoids the slow-mounted-filesystem problem from the previous dataset).

Run this once per session.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# ===== PROJECT ROOT: everything we create on Drive (zips, checkpoints, later outputs) =====
# lives inside this one folder so it's easy to track. Matches your actual Drive layout:
# My Drive > URI-CADS dataset > bbdd_kidney > {images, labels, masks}
DRIVE_PROJECT_ROOT = '/content/drive/MyDrive/URI-CADS dataset'

# The raw dataset itself, inside the project root
DRIVE_DATASET_PATH = f'{DRIVE_PROJECT_ROOT}/bbdd_kidney'

# Everything we generate from here on (zip cache, checkpoints, etc.) also nests under the project root
DRIVE_ZIP_PATH = f'{DRIVE_PROJECT_ROOT}/uri_cads_archive.zip'
DRIVE_CHECKPOINTS_PATH = f'{DRIVE_PROJECT_ROOT}/checkpoints'

LOCAL_ZIP_PATH = '/content/uri_cads_archive.zip'
LOCAL_DATA_DIR = '/content/uri_cads_data'

import os
os.makedirs(DRIVE_CHECKPOINTS_PATH, exist_ok=True)

print('Project root exists:', os.path.isdir(DRIVE_PROJECT_ROOT))
print('Dataset folder exists:', os.path.isdir(DRIVE_DATASET_PATH))
if os.path.isdir(DRIVE_DATASET_PATH):
    print('Top-level contents of bbdd_kidney:', os.listdir(DRIVE_DATASET_PATH))
    for sub in ('images', 'labels', 'masks'):
        sub_path = os.path.join(DRIVE_DATASET_PATH, sub)
        if os.path.isdir(sub_path):
            n_files = len(os.listdir(sub_path))
            print(f'  {sub}/  -> {n_files} files')
        else:
            print(f'  {sub}/  -> NOT FOUND (check folder name/casing)')

Project root exists: True
Dataset folder exists: True
Top-level contents of bbdd_kidney: ['masks', 'images', 'labels']
  images/  -> 1985 files
  labels/  -> 1985 files
  masks/  -> 1985 files


In [3]:
# ONE-TIME STEP: zip the dataset on Drive itself (Drive -> Drive, no local disk involved).
# This is slow the FIRST time (thousands of small files), but only needs to run once, ever.
# Every future session then just copies one zip file (fast) instead of re-touching every image.
#
# IMPORTANT: we zip RELATIVE to the dataset's parent folder (using cwd=), not the absolute path.
# Zipping an absolute path bakes the full /content/drive/MyDrive/... chain into the archive,
# which is what caused the nested content/drive/MyDrive/... folders you saw after extraction.

import os

FORCE_REZIP = False  # set True once to overwrite a bad/old zip; set back to False after a clean run

if os.path.exists(DRIVE_ZIP_PATH) and FORCE_REZIP:
    print('Removing existing (bad) zip before recreating it correctly...')
    os.remove(DRIVE_ZIP_PATH)

if os.path.exists(DRIVE_ZIP_PATH):
    print('Zip already exists on Drive — skipping creation.')
    print('Path:', DRIVE_ZIP_PATH, '| Size (MB):', round(os.path.getsize(DRIVE_ZIP_PATH)/1e6, 1))
else:
    print('Creating zip now — this may take a while the first time.')
    import subprocess, time
    base_dir = os.path.dirname(DRIVE_DATASET_PATH)   # .../URI-CADS dataset
    folder_name = os.path.basename(DRIVE_DATASET_PATH)  # bbdd_kidney
    t0 = time.time()
    # -r recursive, -q quiet; cwd= makes paths inside the zip start at 'bbdd_kidney/...'
    subprocess.run(['zip', '-r', '-q', DRIVE_ZIP_PATH, folder_name], cwd=base_dir)
    print(f'Done in {time.time()-t0:.1f}s. Zip saved to {DRIVE_ZIP_PATH}')

Removing existing (bad) zip before recreating it correctly...
Creating zip now — this may take a while the first time.
Done in 2643.2s. Zip saved to /content/drive/MyDrive/URI-CADS dataset/uri_cads_archive.zip


In [4]:
# EVERY SESSION: copy the single zip file to local disk (fast), then unzip locally (fast, local I/O).
import shutil, os, zipfile, time

os.makedirs(LOCAL_DATA_DIR, exist_ok=True)

t0 = time.time()
shutil.copy(DRIVE_ZIP_PATH, LOCAL_ZIP_PATH)
print(f'Copied zip to local disk in {time.time()-t0:.1f}s')

t0 = time.time()
with zipfile.ZipFile(LOCAL_ZIP_PATH, 'r') as zf:
    zf.extractall(LOCAL_DATA_DIR)
print(f'Unzipped in {time.time()-t0:.1f}s')

# With the fixed zip step, this should now be LOCAL_DATA_DIR/bbdd_kidney/{images,labels,masks}
LOCAL_DATASET_DIR = os.path.join(LOCAL_DATA_DIR, 'bbdd_kidney')
print('Local data dir contents:', os.listdir(LOCAL_DATA_DIR))
if os.path.isdir(LOCAL_DATASET_DIR):
    print('bbdd_kidney/ contents:', os.listdir(LOCAL_DATASET_DIR))
else:
    print('WARNING: bbdd_kidney not found directly under LOCAL_DATA_DIR — check structure above.')

Copied zip to local disk in 0.1s
Unzipped in 1.5s
Local data dir contents: ['bbdd_kidney']
bbdd_kidney/ contents: ['images', 'masks', 'labels']


# Phase 1 — Data Exploration & Ingestion

**We don't yet know URI-CADS's exact annotation format** (JSON manifest? per-image files? CSV? pickle?). This section discovers it programmatically instead of guessing.

Run the next two cells, then **copy the printed output back into the chat** so the mask/bbox parser can be written against the real schema instead of an assumption.

In [5]:
# Print a directory tree (depth-limited so it's readable even with thousands of image files)
import os

def print_tree(root, max_depth=3, max_items_per_dir=15):
    root = root.rstrip('/')
    root_depth = root.count(os.sep)
    for dirpath, dirnames, filenames in os.walk(root):
        depth = dirpath.count(os.sep) - root_depth
        if depth > max_depth:
            dirnames[:] = []  # stop descending further
            continue
        indent = '  ' * depth
        print(f'{indent}{os.path.basename(dirpath)}/')
        sub_indent = '  ' * (depth + 1)
        shown = filenames[:max_items_per_dir]
        for f in shown:
            print(f'{sub_indent}{f}')
        if len(filenames) > max_items_per_dir:
            print(f'{sub_indent}... ({len(filenames) - max_items_per_dir} more files)')

print_tree(LOCAL_DATA_DIR, max_depth=3, max_items_per_dir=15)

uri_cads_data/
  bbdd_kidney/
    images/
      1267.jpg
      175.jpg
      182.jpg
      628.jpg
      272.jpg
      1314.jpg
      217.jpg
      1525.jpg
      564.jpg
      1831.jpg
      635.jpg
      271.jpg
      522.jpg
      1743.jpg
      1143.jpg
      ... (1970 more files)
    masks/
      48.mat
      286.mat
      1142.mat
      1106.mat
      1165.mat
      473.mat
      293.mat
      1976.mat
      1312.mat
      1825.mat
      1606.mat
      1547.mat
      1039.mat
      1243.mat
      139.mat
      ... (1970 more files)
    labels/
      1211.txt
      1575.txt
      1096.txt
      1463.txt
      1635.txt
      312.txt
      896.txt
      565.txt
      1923.txt
      1473.txt
      451.txt
      1379.txt
      750.txt
      346.txt
      1571.txt
      ... (1970 more files)


In [6]:
# Find likely annotation files by extension, and preview their raw content.
# This handles the common possibilities (json/csv/xml/pkl/mat) without assuming which one it is.

import os

candidate_exts = ('.json', '.csv', '.xml', '.pkl', '.pickle', '.mat', '.txt')
found = []
for dirpath, _, filenames in os.walk(LOCAL_DATA_DIR):
    for f in filenames:
        if f.lower().endswith(candidate_exts):
            found.append(os.path.join(dirpath, f))

print(f'Found {len(found)} candidate annotation-like files. Showing up to 5 distinct extensions:\n')

seen_exts = set()
for path in found:
    ext = os.path.splitext(path)[1].lower()
    if ext in seen_exts:
        continue
    seen_exts.add(ext)
    print('='*80)
    print('FILE:', path)
    print('SIZE (bytes):', os.path.getsize(path))
    print('-'*80)
    try:
        if ext == '.json':
            import json
            with open(path) as fh:
                data = json.load(fh)
            if isinstance(data, dict):
                print('Top-level keys:', list(data.keys())[:20])
                # show one nested sample if it's a dict-of-dicts or dict-of-lists
                first_key = next(iter(data))
                print(f'Sample under key "{first_key}":')
                print(str(data[first_key])[:800])
            elif isinstance(data, list):
                print('List length:', len(data))
                print('First item:', str(data[0])[:800])
        elif ext == '.csv':
            with open(path) as fh:
                for i, line in enumerate(fh):
                    print(line.strip())
                    if i >= 5:
                        break
        elif ext == '.pkl' or ext == '.pickle':
            import pickle
            with open(path, 'rb') as fh:
                data = pickle.load(fh)
            print('Type:', type(data))
            print('Preview:', str(data)[:800])
        elif ext == '.mat':
            import scipy.io
            data = scipy.io.loadmat(path)
            print('Keys:', list(data.keys()))
        else:
            with open(path, 'r', errors='ignore') as fh:
                print(fh.read(800))
    except Exception as e:
        print('Could not auto-preview this file:', e)
    print()

if not found:
    print('No obvious annotation files found by extension. Check the tree above manually —')
    print('annotation data may be embedded in filenames, or in a format not listed here.')

Found 3970 candidate annotation-like files. Showing up to 5 distinct extensions:

FILE: /content/uri_cads_data/bbdd_kidney/masks/48.mat
SIZE (bytes): 1885
--------------------------------------------------------------------------------
Keys: ['__header__', '__version__', '__globals__', 'mask']

FILE: /content/uri_cads_data/bbdd_kidney/labels/1211.txt
SIZE (bytes): 77
--------------------------------------------------------------------------------
Image filename: 1211.jpg
Global diagnosis: 0
Global pathologies: 0 0
View: 0




## Confirmed schema so far

- **Masks**: `masks/<id>.mat`, precomputed binary array under the `mask` key — no polygon rasterization needed, just load directly.
- **Labels**: `labels/<id>.txt`, plain `Key: value` lines — `Image filename`, `Global diagnosis` (0/1), `Global pathologies` (space-separated indicators), `View` (0/1).
- **Images**: `images/<id>.jpg` (assumed — confirm below).

The sample label file we've seen (`1211.txt`) has `Global diagnosis: 0`, i.e. a **healthy** case, so it has no local-lesion lines. We still need to see a **pathological** example's label file to learn how bounding boxes are represented for local lesions — that's flagged at the bottom of this section.

In [7]:
# Mask loader: URI-CADS kidney masks are precomputed .mat files, not polygons.
import scipy.io
import numpy as np

def load_kidney_mask(mat_path):
    """Load a precomputed binary kidney mask from a .mat file."""
    data = scipy.io.loadmat(mat_path)
    mask = data['mask']
    return np.asarray(mask).astype(np.uint8)

# quick sanity check on the one sample we've seen
sample_mask = load_kidney_mask(os.path.join(LOCAL_DATASET_DIR, 'masks', '48.mat'))
print('Mask shape:', sample_mask.shape, '| dtype:', sample_mask.dtype, '| unique values:', np.unique(sample_mask))

Mask shape: (459, 716) | dtype: uint8 | unique values: [0 1]


In [8]:
# Label parser: plain 'Key: value' text files.
# Captures any lines beyond the 4 known keys into `extra_lines` so we don't silently drop
# local-lesion/bounding-box info once we learn its format.

def parse_label_file(txt_path):
    known = {}
    extra_lines = []
    with open(txt_path, 'r') as fh:
        for line in fh:
            line = line.strip()
            if not line:
                continue
            if ':' in line:
                key, value = line.split(':', 1)
                key = key.strip()
                value = value.strip()
            else:
                key, value = None, None

            if key == 'Image filename':
                known['image_filename'] = value
            elif key == 'Global diagnosis':
                known['global_diagnosis'] = int(value)
            elif key == 'Global pathologies':
                known['global_pathologies'] = [int(v) for v in value.split()]
            elif key == 'View':
                known['view'] = int(value)
            else:
                extra_lines.append(line)

    known['extra_lines'] = extra_lines
    return known

sample_label = parse_label_file(os.path.join(LOCAL_DATASET_DIR, 'labels', '1211.txt'))
print(sample_label)

{'image_filename': '1211.jpg', 'global_diagnosis': 0, 'global_pathologies': [0, 0], 'view': 0, 'extra_lines': []}


In [9]:
# Build a manifest across the whole dataset: one row per case, joining images/labels/masks by ID.
import pandas as pd

labels_dir = os.path.join(LOCAL_DATASET_DIR, 'labels')
images_dir = os.path.join(LOCAL_DATASET_DIR, 'images')
masks_dir = os.path.join(LOCAL_DATASET_DIR, 'masks')

rows = []
for fname in sorted(os.listdir(labels_dir)):
    if not fname.endswith('.txt'):
        continue
    case_id = os.path.splitext(fname)[0]
    label_path = os.path.join(labels_dir, fname)
    parsed = parse_label_file(label_path)

    image_path = os.path.join(images_dir, parsed.get('image_filename', f'{case_id}.jpg'))
    mask_path = os.path.join(masks_dir, f'{case_id}.mat')

    rows.append({
        'case_id': case_id,
        'image_path': image_path,
        'mask_path': mask_path,
        'image_exists': os.path.exists(image_path),
        'mask_exists': os.path.exists(mask_path),
        'global_diagnosis': parsed.get('global_diagnosis'),
        'global_pathologies': parsed.get('global_pathologies'),
        'view': parsed.get('view'),
        'num_extra_lines': len(parsed.get('extra_lines', [])),
        'extra_lines': parsed.get('extra_lines'),
    })

manifest = pd.DataFrame(rows)
print('Total cases:', len(manifest))
print('Missing images:', (~manifest['image_exists']).sum())
print('Missing masks:', (~manifest['mask_exists']).sum())
print()
print('Global diagnosis distribution (0=healthy, 1=pathological, presumed):')
print(manifest['global_diagnosis'].value_counts())
print()
print('View distribution (0=transversal, 1=longitudinal, presumed):')
print(manifest['view'].value_counts())
print()
print('Cases with extra (unrecognized) label lines — likely local-lesion/bbox info:', (manifest['num_extra_lines'] > 0).sum())
manifest.head()

Total cases: 1985
Missing images: 0
Missing masks: 0

Global diagnosis distribution (0=healthy, 1=pathological, presumed):
global_diagnosis
1    1535
0     450
Name: count, dtype: int64

View distribution (0=transversal, 1=longitudinal, presumed):
view
0    1842
1     143
Name: count, dtype: int64

Cases with extra (unrecognized) label lines — likely local-lesion/bbox info: 1323


,case_id,image_path,mask_path,image_exists,mask_exists,global_diagnosis,global_pathologies,view,num_extra_lines,extra_lines
0,1,/content/uri_cads_data/bbdd_kidney/images/1.jpg,/content/uri_cads_data/bbdd_kidney/masks/1.mat,True,True,1,"[0, 0]",0,1,[Bounding box 1: 326 196 48 32 5]
1,10,/content/uri_cads_data/bbdd_kidney/images/10.jpg,/content/uri_cads_data/bbdd_kidney/masks/10.mat,True,True,1,"[1, 0]",0,0,[]
2,100,/content/uri_cads_data/bbdd_kidney/images/100.jpg,/content/uri_cads_data/bbdd_kidney/masks/100.mat,True,True,1,"[0, 0]",0,2,"[Bounding box 1: 325 180 35 72 1, Bounding box..."
3,1000,/content/uri_cads_data/bbdd_kidney/images/1000...,/content/uri_cads_data/bbdd_kidney/masks/1000.mat,True,True,1,"[0, 0]",0,4,"[Bounding box 1: 375 168 80 58 7, Bounding box..."
4,1001,/content/uri_cads_data/bbdd_kidney/images/1001...,/content/uri_cads_data/bbdd_kidney/masks/1001.mat,True,True,1,"[1, 1]",0,2,"[Bounding box 1: 384 194 46 40 2, Bounding box..."


## One thing still needed to finish Phase 1

The cell above counts how many label files have **extra, unrecognized lines** — those are almost certainly the local-lesion bounding boxes and pathology IDs for non-healthy cases (`Global diagnosis: 1`).

Run this small follow-up and paste back the output:

```python
example = manifest[manifest['num_extra_lines'] > 0].iloc[0]
print(example['case_id'])
print(example['extra_lines'])
```

Once I see the raw format of those extra lines (bounding box coordinates + pathology ID, in whatever order/delimiter they use), I'll finish:

- `parse_local_lesions()` — turns those lines into `(x_min, y_min, x_max, y_max, pathology_id)` tuples
- `bbox_to_coarse_mask()` — rasterizes each box into a binary mask for the coarse-segmentation head
- The PyTorch `Dataset` class tying image + kidney mask + global labels + local boxes together
- The class-balance / shortcut sanity check on this dataset

That completes Phase 1.

In [17]:
import torch
from torch.utils.data import Dataset
from PIL import Image
import numpy as np


def parse_local_lesions(extra_lines):
    """
    Parses lines like: 'Bounding box 1: 326 196 48 32 5'
    Format confirmed from real data: x_min y_min width height pathology_id (COCO-style box).
    Returns a list of dicts: [{'bbox': (x_min, y_min, x_max, y_max), 'pathology_id': int}, ...]
    pathology_id is currently an opaque integer -- real class-name mapping (cyst/hydronephrosis/
    etc.) is still unconfirmed and can be added later without touching this parsing logic.
    """
    lesions = []
    for line in extra_lines:
        if not line.startswith('Bounding box'):
            continue  # ignore anything else we haven't seen yet, rather than crashing
        _, values = line.split(':', 1)
        nums = [int(v) for v in values.split()]
        if len(nums) != 5:
            continue  # unexpected format -- skip defensively rather than crash the whole dataset
        x, y, w, h, pathology_id = nums
        lesions.append({
            'bbox': (x, y, x + w, y + h),
            'pathology_id': pathology_id,
        })
    return lesions


def bbox_to_coarse_mask(bbox, height, width):
    """Rasterize a single (x_min, y_min, x_max, y_max) box into a binary mask."""
    mask = np.zeros((height, width), dtype=np.uint8)
    x_min, y_min, x_max, y_max = [int(round(v)) for v in bbox]
    mask[y_min:y_max, x_min:x_max] = 1
    return mask


class URICADSDataset(Dataset):
    """
    Returns per sample:
      image              - float tensor, C x H x W, normalized
      kidney_mask        - long tensor, H x W (binary)
      global_diagnosis   - int (0=healthy, 1=pathological)
      global_pathologies - long tensor, fixed-length multi-label vector
      lesion_masks       - long tensor, N x H x W (coarse box-masks; N=0 if none/healthy)
      lesion_ids         - long tensor, N (pathology id per lesion)
      case_id            - str, for debugging/traceability
    """

    def __init__(self, manifest_df, transform=None):
        self.manifest = manifest_df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.manifest)

    def __getitem__(self, idx):
        row = self.manifest.iloc[idx]

        image = Image.open(row['image_path']).convert('RGB')
        image = np.array(image)
        height, width = image.shape[:2]

        kidney_mask = load_kidney_mask(row['mask_path'])

        global_diagnosis = int(row['global_diagnosis'])
        global_pathologies = torch.tensor(row['global_pathologies'], dtype=torch.long)

        lesions = parse_local_lesions(row['extra_lines'])
        if lesions:
            lesion_masks = np.stack([
                bbox_to_coarse_mask(l['bbox'], height, width) for l in lesions
            ])
            lesion_ids = torch.tensor([l['pathology_id'] for l in lesions], dtype=torch.long)
        else:
            lesion_masks = np.zeros((0, height, width), dtype=np.uint8)
            lesion_ids = torch.zeros((0,), dtype=torch.long)

        if self.transform:
            image = self.transform(image)
        else:
            image = torch.from_numpy(image).permute(2, 0, 1).float() / 255.0

        return {
            'image': image,
            'kidney_mask': torch.from_numpy(kidney_mask).long(),
            'global_diagnosis': global_diagnosis,
            'global_pathologies': global_pathologies,
            'lesion_masks': torch.from_numpy(lesion_masks).long(),
            'lesion_ids': lesion_ids,
            'case_id': row['case_id'],
        }


dataset = URICADSDataset(manifest)
sample = dataset[0]
print({k: (v.shape if hasattr(v, 'shape') else v) for k, v in sample.items()})

{'image': torch.Size([3, 498, 716]), 'kidney_mask': torch.Size([498, 716]), 'global_diagnosis': 1, 'global_pathologies': torch.Size([2]), 'lesion_masks': torch.Size([1, 498, 716]), 'lesion_ids': torch.Size([1]), 'case_id': '1'}


In [15]:
example = manifest[manifest['num_extra_lines'] > 0].iloc[0]
print(example['case_id'])
print(example['extra_lines'])

1
['Bounding box 1: 326 196 48 32 5']


In [18]:
# Re-check a pathological case now that parse_local_lesions() is filled in
path_sample = dataset[manifest[manifest['case_id'] == '1'].index[0]]
print('lesion_masks shape:', path_sample['lesion_masks'].shape)
print('lesion_ids:', path_sample['lesion_ids'])

# Dataset-wide lesion stats
lesion_counts = manifest['extra_lines'].apply(lambda lines: len(parse_local_lesions(lines)))
print('\nLesion count distribution across dataset:')
print(lesion_counts.value_counts().sort_index())
print('\nCases with 0 lesions (should roughly match healthy count):', (lesion_counts == 0).sum())

all_pathology_ids = [l['pathology_id'] for lines in manifest['extra_lines'] for l in parse_local_lesions(lines)]
import pandas as pd
print('\nPathology ID distribution (semantic mapping still unconfirmed):')
print(pd.Series(all_pathology_ids).value_counts().sort_index())

lesion_masks shape: torch.Size([1, 498, 716])
lesion_ids: tensor([5])

Lesion count distribution across dataset:
extra_lines
0     662
1     685
2     328
3     188
4      75
5      30
6       5
7       8
8       2
9       1
12      1
Name: count, dtype: int64

Cases with 0 lesions (should roughly match healthy count): 662

Pathology ID distribution (semantic mapping still unconfirmed):
1     110
2    1073
3      92
4     384
5      34
6      11
7     618
8     111
9      45
Name: count, dtype: int64


In [19]:
from sklearn.model_selection import train_test_split

# 70/15/15 split, stratified by global_diagnosis so healthy/pathological ratio is preserved
# in every split -- matches the convention from your original baseline notebook.
SEED = 42

train_df, temp_df = train_test_split(
    manifest, test_size=0.30, random_state=SEED, stratify=manifest['global_diagnosis']
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, random_state=SEED, stratify=temp_df['global_diagnosis']
)

manifest['split'] = 'train'
manifest.loc[val_df.index, 'split'] = 'val'
manifest.loc[test_df.index, 'split'] = 'test'

print(manifest['split'].value_counts())
print('\nClass balance check (global_diagnosis) per split:')
print(manifest.groupby('split')['global_diagnosis'].value_counts(normalize=True).round(3))

split
train    1389
val       298
test      298
Name: count, dtype: int64

Class balance check (global_diagnosis) per split:
split  global_diagnosis
test   1                   0.772
       0                   0.228
train  1                   0.773
       0                   0.227
val    1                   0.775
       0                   0.225
Name: proportion, dtype: float64


In [20]:
# Save the finalized manifest (with split assignment) to Drive, inside the same project folder.
MANIFEST_SAVE_PATH = f'{DRIVE_PROJECT_ROOT}/manifest_with_splits.pkl'
manifest.to_pickle(MANIFEST_SAVE_PATH)
print('Saved manifest to:', MANIFEST_SAVE_PATH)
print('Phase 1 complete. Phase 2 notebook will load this file directly.')

Saved manifest to: /content/drive/MyDrive/URI-CADS dataset/manifest_with_splits.pkl
Phase 1 complete. Phase 2 notebook will load this file directly.
